# 10.7 模型怎麼知道圖片放在哪？

# 第 10 章：圖片如何進入文字模型

前置：embedding、序列與Linear。圖片入口先用16×16合成RGB，不需外部權重；圖片是否被理解必須等語言／視覺基礎能力訓練後測試。程式在tiny_perceptron/multimodal.py。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：一個圖片佔位格，展開成三個向量**

展開 labels 同步加 ignored，再做下一字 shift。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/modal_expand.svg")))

**先想一想：**把3個視覺向量塞入1格，原回答索引還對嗎？

placeholder一格會展開成多個視覺向量。labels必須同步展開為ignored；完成後才shift一次，回答首字仍由前一格預測。

**把直覺寫成數學：**長度變化=N_visual−1；position、attention、labels都要用展開後長度。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.data import ByteTokenizer, IGNORE
from tiny_perceptron.multimodal import expand_modalities

tok = ByteTokenizer()
embedding = nn.Embedding(264, 8)
ids = torch.tensor([1, 5, 4, 73, 2])
labels = torch.tensor([-100, -100, -100, 73, 2])
features = {5: torch.randn(3, 8)}
x, y = expand_modalities(ids, labels, embedding, features, {5})
print(x.shape, y.tolist())
assert y[0, -2].item() == 73 and y.shape[1] == x.shape[1]

**如何讀結果：**ids是未shift序列；[73,2]是答案與EOS。attention依新長度建立，視覺位置不直接算文字loss。

**只改一件事：**只把視覺向量數從3改為5，核對答案索引跟著移動。
